In [1]:
# cell 1
import sys
sys.path.append("..")
from src.data import load_m5, trim_leading_zeros
from src.features import make_features
from src.experiment import run_experiment, summarize

df = trim_leading_zeros(load_m5("../data", store_id="CA_1", cat_id="FOODS"))

In [2]:
# cell 2 - pick 10 series spanning fast to sparse
d3 = df[df["dept_id"] == "FOODS_3"]
stats = d3.groupby("id")["sales"].agg(mean="mean", zero_rate=lambda s: (s == 0).mean())
stats = stats[stats["mean"] > 0.5].sort_values("mean")
picks = stats.iloc[:: max(1, len(stats) // 10)].head(10).index.tolist()
sub = d3[d3["id"].isin(picks)]
print(stats.loc[picks].round(2))

                             mean  zero_rate
id                                          
FOODS_3_539_CA_1_evaluation  0.50       0.73
FOODS_3_802_CA_1_evaluation  0.65       0.59
FOODS_3_010_CA_1_evaluation  0.84       0.51
FOODS_3_743_CA_1_evaluation  1.07       0.50
FOODS_3_733_CA_1_evaluation  1.28       0.52
FOODS_3_813_CA_1_evaluation  1.58       0.39
FOODS_3_632_CA_1_evaluation  1.95       0.26
FOODS_3_236_CA_1_evaluation  2.67       0.34
FOODS_3_661_CA_1_evaluation  3.83       0.05
FOODS_3_476_CA_1_evaluation  5.99       0.14


In [ ]:
# cell 3 - prove the pipeline isn't leaking before trusting any results
from src.validate import run_all
run_all()

,id,n_train,n_test,mean_sales,zero_rate,model_rmse,model_mae,model_wape,model_bias,base_rmse,base_mae,base_wape,base_bias,won,pct_improvement
0,FOODS_3_010_CA_1_evaluation,1052,28,0.83,0.51,0.93,0.78,1.22,-0.04,1.15,0.68,1.06,-0.04,True,18.89
1,FOODS_3_236_CA_1_evaluation,1157,28,2.62,0.35,2.48,1.96,0.62,-0.50,2.73,2.04,0.65,-0.68,True,9.27
2,FOODS_3_476_CA_1_evaluation,1878,28,6.08,0.14,3.21,2.52,0.92,1.77,2.92,2.39,0.87,0.32,False,-9.95
3,FOODS_3_539_CA_1_evaluation,1647,28,0.49,0.74,0.90,0.75,1.11,-0.02,1.25,1.00,1.47,0.00,True,28.33
4,FOODS_3_632_CA_1_evaluation,1878,28,1.97,0.25,2.46,1.67,0.78,-0.19,3.61,2.64,1.23,0.43,True,31.73
5,FOODS_3_661_CA_1_evaluation,695,28,3.87,0.05,1.76,1.38,0.35,0.32,2.43,2.00,0.51,-0.36,True,27.65
6,FOODS_3_733_CA_1_evaluation,800,28,1.20,0.54,1.24,1.02,1.50,0.72,1.35,0.96,1.42,-0.04,True,8.38
7,FOODS_3_743_CA_1_evaluation,464,28,1.04,0.51,1.91,1.54,0.74,0.27,2.39,1.86,0.90,0.00,True,20.07
8,FOODS_3_802_CA_1_evaluation,1850,28,0.66,0.59,0.55,0.46,1.18,0.17,0.73,0.46,1.18,0.11,True,24.49
9,FOODS_3_813_CA_1_evaluation,653,28,1.55,0.40,2.05,1.44,0.54,-0.67,2.18,1.82,0.68,-0.39,True,6.05


In [ ]:
# cell 4
summarize(res)

In [ ]:
# Cell 5
import pandas as pd
from src.features import make_features
from src.evaluate import train_test_split_by_date

f = make_features(sub, horizon=7)
g = f[f["id"] == f["id"].iloc[0]].sort_values("date")

for td in (7, 14, 28):
    tr, te = train_test_split_by_date(g, test_days=td)
    leaks = ((te["date"] - pd.Timedelta(days=7)) > tr["date"].max()).sum()
    print(f"test_days={td:2d} -> {leaks:2d} of {len(te)} rows leak")

In [ ]:
# Cell 6
from src.experiment import run_walk_forward, summarize, summarize_by_series

res = run_walk_forward(f, horizon=7, n_folds=8)
print(summarize(res))
summarize_by_series(res).round(2)

{'series': 80, 'win_rate_pct': np.float64(75.0), 'mean_improvement_pct': np.float64(-inf), 'median_improvement_pct': np.float64(18.0), 'q1_improvement_pct': np.float64(1.5), 'q3_improvement_pct': np.float64(33.5)}


,folds,mean_sales,zero_rate,model_rmse,base_rmse,win_rate,mean_improvement
id,,,,,,,
FOODS_3_661_CA_1_evaluation,8,3.87,0.05,2.25,3.20,0.88,25.27
FOODS_3_476_CA_1_evaluation,8,6.09,0.14,2.73,2.92,0.50,5.58
FOODS_3_632_CA_1_evaluation,8,1.96,0.25,1.96,2.83,0.75,27.49
FOODS_3_236_CA_1_evaluation,8,2.62,0.35,2.20,2.45,0.75,4.89
FOODS_3_813_CA_1_evaluation,8,1.55,0.40,2.04,2.53,0.62,11.24
FOODS_3_010_CA_1_evaluation,8,0.82,0.51,0.94,1.36,1.00,28.00
FOODS_3_743_CA_1_evaluation,8,1.04,0.51,1.86,2.15,0.75,12.45
FOODS_3_733_CA_1_evaluation,8,1.20,0.53,1.27,1.29,0.62,-0.08
FOODS_3_802_CA_1_evaluation,8,0.66,0.59,0.41,0.48,0.62,-inf
